## Generate main trial table (Experiment 1 replication)
Balanced 2×2 (PRIME × IDENTITY) vMMR trial generator

In [ ]:
from pathlib import Path
import random
import pandas as pd

# --- Participant / seeding -------------------------------------------------
PARTICIPANT_ID = 1          # <-- set this per participant
BASE_SEED      = 42
seed = BASE_SEED + PARTICIPANT_ID     # per-participant. For the original fixed
                                      # design, use: seed = BASE_SEED
rng = random.Random(seed)

# --- Resolve project paths -------------------------------------------------
def find_project_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'stimuli' / 'words' / 'death_words.csv').exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find the project root starting from {start_path}. "
        "Expected stimuli/words/death_words.csv in the workspace tree."
    )

PROJECT_DIR = find_project_root(Path.cwd())
WORDS_DIR   = PROJECT_DIR / 'stimuli' / 'words'
FACES_DIR   = PROJECT_DIR / 'stimuli' / 'faces' / 'GenAI_faces'
print(f'Using project directory: {PROJECT_DIR}')
print(f'Seed: {seed}  (participant {PARTICIPANT_ID})')

# --- Parameters ------------------------------------------------------------
N_BLOCKS             = 3
TRIALS_PER_BLOCK     = 120
TARGET_PER_BLOCK     = 30
NONTARGET_PER_BLOCK  = TRIALS_PER_BLOCK - TARGET_PER_BLOCK   # 90
CONDITIONS   = ['DS', 'DO', 'NS', 'NO']                     # death/self, death/other, neg/self, neg/other
DIAGONALS    = [('DS', 'NO'), ('DO', 'NS')]                 # each pair: 1 death+1 neg, 1 self+1 other
PRIME_MAP    = {'DS': 'DEATH', 'DO': 'DEATH', 'NS': 'NEGATIVE', 'NO': 'NEGATIVE'}
IDENTITY_MAP = {'DS': 'SELF',  'DO': 'OTHER', 'NS': 'SELF',     'NO': 'OTHER'}

STD_IMAGE     = {'SELF': str(FACES_DIR / 'self.png'),  'OTHER': str(FACES_DIR / 'other.png')}
TGT_IMAGE     = {'SELF': str(FACES_DIR / 'self_target.png'), 'OTHER': str(FACES_DIR / 'other_target.png')}
DEV_IMAGE     = str(FACES_DIR / 'morph50.png')
DEV_TGT_IMAGE = str(FACES_DIR / 'morph50_target.png')

death_words    = pd.read_csv(WORDS_DIR / 'death_words.csv')['word'].tolist()
negative_words = pd.read_csv(WORDS_DIR / 'negative_words.csv')['word'].tolist()

# --- Helpers ---------------------------------------------------------------
def balanced_split(total, rng):
    """Split `total` across the 4 conditions as evenly as possible; any remainder
    of 2 goes to a random diagonal pair (preserving exact PRIME & IDENTITY balance)."""
    base, rem = divmod(total, 4)
    counts = {c: base for c in CONDITIONS}
    if rem == 2:
        for c in rng.choice(DIAGONALS):
            counts[c] += 1
    elif rem != 0:                      # not expected for 90/30, kept for safety
        for c in rng.sample(CONDITIONS, rem):
            counts[c] += 1
    return counts

def make_word_cycle(words, n_needed, rng):
    """Cycle through the word list, reshuffling each pass, to fill n_needed slots."""
    result = []
    while len(result) < n_needed:
        s = words[:]
        rng.shuffle(s)
        result.extend(s)
    return result[:n_needed]

# --- Build trial tokens per block (order-shuffled within block) ------------
rows = []
for block in range(1, N_BLOCKS + 1):
    nt_counts = balanced_split(NONTARGET_PER_BLOCK, rng)   # 45/45 PRIME, 45/45 IDENTITY
    tg_counts = balanced_split(TARGET_PER_BLOCK,    rng)   # 15/15 PRIME, 15/15 IDENTITY

    tokens = []
    for c in CONDITIONS:
        tokens += [(c, 0)] * nt_counts[c]
        tokens += [(c, 1)] * tg_counts[c]
    rng.shuffle(tokens)

    for j, (cond, is_target) in enumerate(tokens, start=1):
        rows.append({
            'block':          block,
            'trial_in_block': j,
            'condCode':       cond,
            'primeType':      PRIME_MAP[cond],
            'identity':       IDENTITY_MAP[cond],
            'isTarget':       is_target,
        })

# --- Assign nStandards from a balanced per-condition pool ------------------
length_cursor = {c: 0 for c in CONDITIONS}
length_pool   = {}
for c in CONDITIONS:
    n_c  = sum(1 for r in rows if r['condCode'] == c)
    pool = ([3, 4, 5, 6] * (n_c // 4 + 1))[:n_c]
    rng.shuffle(pool)
    length_pool[c] = pool
for r in rows:
    c = r['condCode']
    r['nStandards'] = length_pool[c][length_cursor[c]]
    length_cursor[c] += 1

# --- Target position: 1..n_std = standard slot, n_std+1 = deviant slot -----
for r in rows:
    r['targetPosition'] = rng.randint(1, r['nStandards'] + 1) if r['isTarget'] else -1

# --- Assign words (exactly even: 180 trims / 36 words = 5 each) ------------
death_idx = [i for i, r in enumerate(rows) if r['primeType'] == 'DEATH']
neg_idx   = [i for i, r in enumerate(rows) if r['primeType'] == 'NEGATIVE']
death_cycle = make_word_cycle(death_words,    len(death_idx), rng)
neg_cycle   = make_word_cycle(negative_words, len(neg_idx),   rng)
for k, i in enumerate(death_idx):
    rows[i]['word'] = death_cycle[k]
for k, i in enumerate(neg_idx):
    rows[i]['word'] = neg_cycle[k]

# --- Image columns ---------------------------------------------------------
for r in rows:
    ident = r['identity']
    r['standardImage']       = STD_IMAGE[ident]
    r['deviantImage']        = DEV_IMAGE
    r['standardTargetImage'] = TGT_IMAGE[ident]
    r['deviantTargetImage']  = DEV_TGT_IMAGE

# --- Assemble (preserve original column order / Coder-loop contract) -------
COLS = ['block', 'trial_in_block', 'condCode', 'primeType', 'identity', 'word',
        'nStandards', 'isTarget', 'targetPosition', 'standardImage', 'deviantImage',
        'standardTargetImage', 'deviantTargetImage']
df = pd.DataFrame(rows)[COLS]

# --- Inline sanity checks (full audit = the diagnostic cell) ---------------
nt = df[df.isTarget == 0]
tg = df[df.isTarget == 1]
assert len(df) == 360 and len(tg) == 90 and len(nt) == 270
assert (df.groupby('block').size() == 120).all()
assert (df.primeType == 'DEATH').sum() == (df.primeType == 'NEGATIVE').sum() == 180
assert (nt.primeType == 'DEATH').sum() == (nt.primeType == 'NEGATIVE').sum() == 135
assert (df.identity == 'SELF').sum() == (df.identity == 'OTHER').sum() == 180
assert df.nStandards.min() >= 3
print("Sanity checks passed: 360 trials, 120/block, PRIME 180/180 & 135/135, "
      "IDENTITY 180/180, nStandards>=3.")
print("Per-condition totals:", df.condCode.value_counts().sort_index().to_dict())

# --- Save ------------------------------------------------------------------
output_path = PROJECT_DIR / 'conditions' / 'main_trials.csv'
df.to_csv(output_path, index=False, encoding='utf-8')
print(f"Saved to {output_path}")

Using project directory: /Users/omertzuk/code/EEG/vMMR_EEG_PsychoPy_experiment
Seed: 43  (participant 1)
Sanity checks passed: 360 trials, 120/block, PRIME 180/180 & 135/135, IDENTITY 180/180, nStandards>=3.
Per-condition totals: {'DO': 88, 'DS': 92, 'NO': 92, 'NS': 88}
Saved to /Users/omertzuk/code/EEG/vMMR_EEG_PsychoPy_experiment/conditions/main_trials.csv


In [2]:
from pathlib import Path
import pandas as pd
import random

random.seed(99)

# --- Resolve project paths ---
def find_project_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'stimuli' / 'words' / 'death_words.csv').exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find the project root starting from {start_path}. "
        "Expected to find stimuli/words/death_words.csv in the workspace tree."
    )

PROJECT_DIR = find_project_root(Path.cwd())
WORDS_DIR = PROJECT_DIR / 'stimuli' / 'words'
# FACES_DIR = PROJECT_DIR / 'stimuli' / 'faces'
FACES_DIR = PROJECT_DIR / 'stimuli' / 'faces' / 'GenAI_faces'

print(f'Using project directory: {PROJECT_DIR}')

CONDITIONS = ['DS', 'DO', 'NS', 'NO']
PRIME_MAP    = {'DS': 'DEATH', 'DO': 'DEATH', 'NS': 'NEGATIVE', 'NO': 'NEGATIVE'}
IDENTITY_MAP = {'DS': 'SELF',  'DO': 'OTHER', 'NS': 'SELF',     'NO': 'OTHER'}
STD_IMAGE    = {'SELF': str(FACES_DIR / 'self.png'), 'OTHER': str(FACES_DIR / 'other.png')}
TGT_IMAGE    = {'SELF': str(FACES_DIR / 'self_target.png'), 'OTHER': str(FACES_DIR / 'other_target.png')}
DEV_IMAGE        = str(FACES_DIR / 'morph50.png')
DEV_TGT_IMAGE    = str(FACES_DIR / 'morph50_target.png')

death_words = pd.read_csv(WORDS_DIR / 'death_words.csv')['word'].tolist()
negative_words = pd.read_csv(WORDS_DIR / 'negative_words.csv')['word'].tolist()

# 16 trials: each condition x4, 4 of them targets (one per condition)
conds = CONDITIONS * 4
random.shuffle(conds)
target_conds = set(random.sample(CONDITIONS, 4))  # one target per condition
target_assigned = {c: False for c in CONDITIONS}

rows = []
d_idx, n_idx = 0, 0

for i, cond in enumerate(conds):
    identity = IDENTITY_MAP[cond]
    n_std = random.randint(3, 6)
    
    is_target = (not target_assigned[cond]) and (cond in target_conds)
    if is_target:
        target_assigned[cond] = True
        target_pos = random.randint(1, n_std + 1)
    else:
        target_pos = -1

    if PRIME_MAP[cond] == 'DEATH':
        word = death_words[d_idx % len(death_words)]
        d_idx += 1
    else:
        word = negative_words[n_idx % len(negative_words)]
        n_idx += 1

    rows.append({
        'block': 0,
        'trial_in_block': i + 1,
        'condCode': cond,
        'primeType': PRIME_MAP[cond],
        'identity': identity,
        'word': word,
        'nStandards': n_std,
        'isTarget': int(is_target),
        'targetPosition': target_pos,
        'standardImage': STD_IMAGE[identity],
        'deviantImage': DEV_IMAGE,
        'standardTargetImage': TGT_IMAGE[identity],
        'deviantTargetImage': DEV_TGT_IMAGE,
    })


df_prac = pd.DataFrame(rows)
print(df_prac[['condCode','primeType','identity','word','nStandards','isTarget','targetPosition']])

output_path = PROJECT_DIR / 'conditions' / 'practice_trials.csv'
df_prac.to_csv(output_path, index=False, encoding='utf-8')
print(f"\nSaved: {output_path}")

Using project directory: /Users/omertzuk/code/EEG/vMMR_EEG_PsychoPy_experiment
   condCode primeType identity              word  nStandards  isTarget  \
0        DS     DEATH     SELF             אֵבֶל           6         1   
1        NO  NEGATIVE    OTHER            אֲסִיר           4         1   
2        NS  NEGATIVE     SELF         בְּחִילָה           6         1   
3        DS     DEATH     SELF          אוֹבְדָן           4         0   
4        NO  NEGATIVE    OTHER          בִּיזוּי           5         0   
5        DO     DEATH    OTHER        אַזְכָּרָה           3         1   
6        DS     DEATH     SELF           אֵיידְס           6         0   
7        DO     DEATH    OTHER             אֵפֶר           5         0   
8        NO  NEGATIVE    OTHER            גֶּזֶל           6         0   
9        NS  NEGATIVE     SELF  גִילוּי עֲרָיוֹת           3         0   
10       DO     DEATH    OTHER          גִ'יהַאד           5         0   
11       NS  NEGATIVE     SELF   

## Diagnosis cell

In [8]:
# %%
# ============================================================
# Diagnostic: audit conditions/main_trials.csv  (READ-ONLY)
# Verifies Experiment-1 design constraints and surfaces balance issues.
# ============================================================
from pathlib import Path
import pandas as pd
import numpy as np

def find_project_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'stimuli' / 'words' / 'death_words.csv').exists():
            return candidate
    raise FileNotFoundError(f"Project root not found from {start_path}")

PROJECT_DIR = find_project_root(Path.cwd())
CSV_PATH    = PROJECT_DIR / 'conditions' / 'main_trials.csv'
WORDS_DIR   = PROJECT_DIR / 'stimuli' / 'words'

df = pd.read_csv(CSV_PATH, encoding='utf-8')
death_words    = set(pd.read_csv(WORDS_DIR / 'death_words.csv')['word'].tolist())
negative_words = set(pd.read_csv(WORDS_DIR / 'negative_words.csv')['word'].tolist())

nt = df[df.isTarget == 0]   # non-target = the sequences that enter the ERF analysis
tg = df[df.isTarget == 1]

L = "=" * 60
print(L); print(f"TRIAL TABLE DIAGNOSTIC: {CSV_PATH}"); print(L)

# ---- 1. Gross counts ----
print("\n[1] GROSS COUNTS")
print(f"  total trials      : {len(df):>4}   (expect 360)")
print(f"  target trials     : {len(tg):>4}   (expect  90)")
print(f"  non-target trials : {len(nt):>4}   (expect 270)")
print(f"  blocks            : {sorted(df.block.unique())}")

# ---- 2. Condition x target-status ----
print("\n[2] CONDITION x TARGET STATUS")
ct = pd.crosstab(df.condCode, df.isTarget, margins=True)
ct.columns = [{0:'nonTarget',1:'target'}.get(c, c) for c in ct.columns]
print(ct.to_string())
print("  NOTE: non-target count per condition = # analyzable MMR sequences per condition")

# ---- 3. PRIME balance (the flagged issue) ----
print("\n[3] PRIME BALANCE  (death vs negative)")
for label, sub in [('all', df), ('non-target', nt), ('target', tg)]:
    d = (sub.primeType == 'DEATH').sum()
    n = (sub.primeType == 'NEGATIVE').sum()
    flag = 'OK' if d == n else f'IMBALANCE (Δ={d-n:+d})'
    print(f"  {label:<11}: DEATH={d:>4}  NEG={n:>4}   {flag}")

# ---- 4. IDENTITY balance (self vs other) ----
print("\n[4] IDENTITY BALANCE  (self vs other)")
for label, sub in [('all', df), ('non-target', nt), ('target', tg)]:
    s = (sub.identity == 'SELF').sum()
    o = (sub.identity == 'OTHER').sum()
    flag = 'OK' if s == o else f'IMBALANCE (Δ={s-o:+d})'
    print(f"  {label:<11}: SELF={s:>4}  OTHER={o:>4}   {flag}")

# ---- 5. Sequence length (nStandards) ----
print("\n[5] SEQUENCE LENGTH  (nStandards, expect uniform 3-6)")
print("  overall distribution:")
print(df.nStandards.value_counts().sort_index().to_string().replace('\n','\n    ').rjust(0))
g = df.groupby('condCode')['nStandards'].agg(['mean','min','max','sum'])
print("  per condition:")
print(g.round(2).to_string().replace('\n','\n    '))
print(f"  global min nStandards = {df.nStandards.min()}  "
      f"({'OK — 3rd-standard rule always satisfiable' if df.nStandards.min()>=3 else 'PROBLEM: <3 breaks 3rd-standard rule'})")

# ---- 6. Standard/deviant image ratio (NON-TARGET only, per paper) ----
print("\n[6] STANDARD / DEVIANT RATIO  (non-target sequences)")
n_std = nt.nStandards.sum()
n_dev = len(nt)                       # one clean deviant per non-target sequence
tot   = n_std + n_dev
print(f"  standards : {n_std:>5}  ({100*n_std/tot:4.1f}%)")
print(f"  deviants  : {n_dev:>5}  ({100*n_dev/tot:4.1f}%)")
print(f"  paper reports ~82% / ~18%")

# ---- 7. Target position ----
print("\n[7] TARGET POSITION  (among target trials)")
at_dev = (tg.targetPosition == tg.nStandards + 1).sum()
at_std = (tg.targetPosition <= tg.nStandards).sum()
print(f"  at a standard slot : {at_std:>3}  ({100*at_std/len(tg):4.1f}%)")
print(f"  at the deviant slot: {at_dev:>3}  ({100*at_dev/len(tg):4.1f}%)")
print(f"  position range     : {tg.targetPosition.min()}..{tg.targetPosition.max()}")

# ---- 8. Word integrity & coverage ----
print("\n[8] WORD INTEGRITY & COVERAGE")
empty = (df.word.astype(str).str.len() == 0).sum()
d_bad = df[(df.primeType=='DEATH')    & (~df.word.isin(death_words))]
n_bad = df[(df.primeType=='NEGATIVE') & (~df.word.isin(negative_words))]
print(f"  empty word cells        : {empty}  ({'OK' if empty==0 else 'PROBLEM'})")
print(f"  DEATH trials w/ non-death word    : {len(d_bad)}  ({'OK' if len(d_bad)==0 else 'PROBLEM'})")
print(f"  NEGATIVE trials w/ non-neg word   : {len(n_bad)}  ({'OK' if len(n_bad)==0 else 'PROBLEM'})")
for label, wordset, sub in [('death', death_words, df[df.primeType=='DEATH']),
                            ('negative', negative_words, df[df.primeType=='NEGATIVE'])]:
    vc = sub.word.value_counts()
    used, total = vc.size, len(wordset)
    print(f"  {label:<8}: {used}/{total} unique words used, "
          f"per-word count min={vc.min()} max={vc.max()} "
          f"({'even' if vc.max()-vc.min()<=1 else f'spread={vc.max()-vc.min()}'})")

# ---- 9. Per-block condition counts (surfaces deterministic skew) ----
print("\n[9] PER-BLOCK CONDITION COUNTS  (non-target)")
print(pd.crosstab(nt.block, nt.condCode).to_string().replace('\n','\n  '))

print("\n" + L); print("END DIAGNOSTIC"); print(L)

TRIAL TABLE DIAGNOSTIC: /Users/omertzuk/code/EEG/vMMR_EEG_PsychoPy_experiment/conditions/main_trials.csv

[1] GROSS COUNTS
  total trials      :  360   (expect 360)
  target trials     :   90   (expect  90)
  non-target trials :  270   (expect 270)
  blocks            : [np.int64(1), np.int64(2), np.int64(3)]

[2] CONDITION x TARGET STATUS
          nonTarget  target  All
condCode                        
DO               66      22   88
DS               69      23   92
NO               69      23   92
NS               66      22   88
All             270      90  360
  NOTE: non-target count per condition = # analyzable MMR sequences per condition

[3] PRIME BALANCE  (death vs negative)
  all        : DEATH= 180  NEG= 180   OK
  non-target : DEATH= 135  NEG= 135   OK
  target     : DEATH=  45  NEG=  45   OK

[4] IDENTITY BALANCE  (self vs other)
  all        : SELF= 180  OTHER= 180   OK
  non-target : SELF= 135  OTHER= 135   OK
  target     : SELF=  45  OTHER=  45   OK

[5] SEQUENCE LEN